# 📊 Fashion Datasets - Exploratory Data Analysis

This notebook performs comprehensive EDA on two fashion datasets:

1. **Clothing Dataset** (Tabular/Metadata) - 5,403 samples, 20 classes
2. **Fashion MNIST** (Images) - 70,000 samples, 10 classes

---

## 📋 Setup

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plotly for interactive plots
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 100)

# Style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("✅ Imports complete")

---

## 1️⃣ CLOTHING DATASET (Tabular/Metadata)

### 1.1 Load Data

In [ ]:
# Load the clothing dataset
data_path = "../data/clothing_dataset.csv"
df = pd.read_csv(data_path)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nFirst 10 rows:")
display(df.head(10))

### 1.2 Basic Statistics & Data Quality

In [ ]:
# Missing values
print("=== MISSING VALUES ===")
print(df.isnull().sum())

# Duplicate rows
print(f"\n=== DUPLICATE ROWS: {df.duplicated().sum()} ===")

# Unique values per column
print("\n=== UNIQUE VALUES ===")
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique")

# Target distribution
print("\n=== CLASS DISTRIBUTION ===")
class_counts = df['label'].value_counts()
display(class_counts.to_frame('Count').assign(Percentage=lambda x: (x['Count'] / len(df) * 100).round(2)))

### 1.3 Class Distribution Visualization

In [ ]:
# Static matplotlib/seaborn
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Count plot
class_counts = df['label'].value_counts()
axes[0].bar(range(len(class_counts)), class_counts.values, color='steelblue', edgecolor='black')
axes[0].set_xticks(range(len(class_counts)))
axes[0].set_xticklabels(class_counts.index, rotation=45, ha='right')
axes[0].set_title('Class Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Samples')
axes[0].set_xlabel('Class')

# Percentage plot
class_pct = (class_counts / len(df) * 100).round(2)
axes[1].bar(range(len(class_pct)), class_pct.values, color='coral', edgecolor='black')
axes[1].set_xticks(range(len(class_pct)))
axes[1].set_xticklabels(class_pct.index, rotation=45, ha='right')
axes[1].set_title('Class Distribution (Percentage)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_xlabel('Class')

plt.tight_layout()
plt.show()

# Interactive Plotly version
fig_plotly = px.bar(
    x=class_counts.index, y=class_counts.values,
    labels={'x': 'Class', 'y': 'Count'},
    title='Fashion Product Class Distribution (Interactive)',
    color=class_counts.values,
    color_continuous_scale='Viridis',
    text=class_counts.values
)
fig_plotly.update_traces(texttemplate='%{text}', textposition='outside')
fig_plotly.show()

### 1.4 Class Imbalance Analysis

In [ ]:
# Imbalance metrics
max_class = class_counts.max()
min_class = class_counts.min()
imbalance_ratio = max_class / min_class

print(f"Majority class: {class_counts.index[0]} ({max_class} samples)")
print(f"Minority class: {class_counts.index[-1]} ({min_class} samples)")
print(f"Imbalance ratio (max/min): {imbalance_ratio:.2f}")
print(f"\nClasses with < 50 samples:")
small_classes = class_counts[class_counts < 50]
for cls, cnt in small_classes.items():
    print(f"  {cls}: {cnt}")

# Visualize imbalance
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(class_counts)), class_counts.values, color='steelblue', edgecolor='black')
ax.axhline(y=max_class, color='red', linestyle='--', label=f'Majority ({max_class})')
ax.axhline(y=min_class, color='orange', linestyle='--', label=f'Minority ({min_class})')
ax.set_xticks(range(len(class_counts)))
ax.set_xticklabels(class_counts.index, rotation=45, ha='right')
ax.set_yscale('log')
ax.set_title('Class Distribution (Log Scale) - Shows Imbalance', fontsize=14, fontweight='bold')
ax.set_ylabel('Count (log scale)')
ax.legend()
plt.tight_layout()
plt.show()

### 1.5 Kids vs Adults Distribution

In [ ]:
# Overall kids distribution
kids_dist = df['kids'].value_counts()
print(f"Overall kids distribution:\n{kids_dist}")
print(f"Kids %: {kids_dist[True]/len(df)*100:.1f}%")

# Kids distribution per class
kids_crosstab = pd.crosstab(df['label'], df['kids'], normalize='index') * 100
kids_crosstab.columns = ['Adults (%)', 'Kids (%')]
kids_crosstab = kids_crosstab.sort_values('Kids (%)', ascending=False)

print("\nKids % by class (sorted):")
display(kids_crosstab.round(1))

# Stacked bar chart
fig, ax = plt.subplots(figsize=(12, 6))
kids_crosstab[['Adults (%)', 'Kids (%)']].plot(
    kind='bar', stacked=True, ax=ax, 
    color=['skyblue', 'lightcoral'], edgecolor='black'
)
ax.set_title('Kids vs Adults Distribution by Class (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Percentage')
ax.set_xlabel('Class')
ax.legend(title='Category')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

### 1.6 Sender Analysis

In [ ]:
# Sender statistics
sender_counts = df['sender_id'].value_counts()

print(f"Unique senders: {df['sender_id'].nunique()}")
print(f"Samples per sender - Mean: {sender_counts.mean():.1f}, Median: {sender_counts.median()}")
print(f"Max samples from one sender: {sender_counts.max()}")
print(f"Min samples from one sender: {sender_counts.min()}")

# Top 20 senders
print("\nTop 20 senders by sample count:")
display(sender_counts.head(20).to_frame('Sample Count'))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Top 20
top_senders = sender_counts.head(20)
axes[0].bar(range(len(top_senders)), top_senders.values, color='mediumseagreen', edgecolor='black')
axes[0].set_xticks(range(len(top_senders)))
axes[0].set_xticklabels(top_senders.index.astype(str), rotation=45, ha='right')
axes[0].set_title('Top 20 Senders by Sample Count', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Samples')
axes[0].set_xlabel('Sender ID')

# Distribution
axes[1].hist(sender_counts.values, bins=50, color='gold', edgecolor='black', alpha=0.7)
axes[1].set_title('Distribution of Samples per Sender', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Number of Samples')
axes[1].set_ylabel('Number of Senders')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

# Sender-class relationship
sender_class = df.groupby('sender_id')['label'].nunique()
print(f"\nClasses per sender - Mean: {sender_class.mean():.2f}, Max: {sender_class.max()}")
print(f"Senders with only 1 class: {(sender_class == 1).sum()}")

### 1.7 Cross Analysis: Sender × Class × Kids

In [ ]:
# Heatmap: Class vs Sender (top 30 senders)
top_30_senders = sender_counts.head(30).index
df_top = df[df['sender_id'].isin(top_30_senders)]

pivot = pd.crosstab(df_top['sender_id'], df_top['label'])

plt.figure(figsize=(16, 10))
sns.heatmap(pivot, cmap='YlOrRd', annot=True, fmt='d', cbar_kws={'label': 'Count'})
plt.title('Sample Count: Sender (Top 30) × Class', fontsize=14, fontweight='bold')
plt.xlabel('Class')
plt.ylabel('Sender ID')
plt.tight_layout()
plt.show()

# Class distribution for top sender vs overall
top_sender_id = sender_counts.index[0]
top_sender_data = df[df['sender_id'] == top_sender_id]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_sender_data['label'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title(f'Class Distribution - Top Sender (ID: {top_sender_id})', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

df['label'].value_counts().plot(kind='bar', ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('Class Distribution - Overall Dataset', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 1.8 Image ID Analysis

In [ ]:
# Image ID characteristics
print(f"Image ID sample: {df['image'].iloc[0]}")
print(f"Image ID length: {len(df['image'].iloc[0])}")
print(f"Unique image IDs: {df['image'].nunique()} (should equal total rows: {len(df)})")

# Check if any image appears multiple times
img_dupes = df['image'].duplicated().sum()
print(f"Duplicate image IDs: {img_dupes}")

---

## 2️⃣ FASHION MNIST (Image Dataset)

### 2.1 Load Data

In [ ]:
# Load Fashion MNIST
import tensorflow as tf

print("Loading Fashion MNIST...")
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

# Class names
FMNIST_CLASSES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

print(f"Train images: {X_train.shape}")
print(f"Train labels: {y_train.shape}")
print(f"Test images: {X_test.shape}")
print(f"Test labels: {y_test.shape}")
print(f"Image dtype: {X_train.dtype}")
print(f"Pixel range: [{X_train.min()}, {X_train.max()}]")

# Class distribution
train_counts = np.bincount(y_train)
test_counts = np.bincount(y_test)

dist_df = pd.DataFrame({
    'Class': FMNIST_CLASSES,
    'Train Count': train_counts,
    'Test Count': test_counts,
    'Total': train_counts + test_counts
})
display(dist_df)

### 2.2 Class Distribution

In [ ]:
# Visualize class balance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(10), train_counts, color='steelblue', edgecolor='black')
axes[0].set_xticks(range(10))
axes[0].set_xticklabels(FMNIST_CLASSES, rotation=45, ha='right')
axes[0].set_title('Training Set Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')

axes[1].bar(range(10), test_counts, color='coral', edgecolor='black')
axes[1].set_xticks(range(10))
axes[1].set_xticklabels(FMNIST_CLASSES, rotation=45, ha='right')
axes[1].set_title('Test Set Class Distribution', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

# Verify perfect balance
print(f"Train balance check: min={train_counts.min()}, max={train_counts.max()}, all equal? {len(set(train_counts)) == 1}")
print(f"Test balance check: min={test_counts.min()}, max={test_counts.max()}, all equal? {len(set(test_counts)) == 1}")

### 2.3 Sample Images per Class

In [ ]:
# Display sample images from each class
fig, axes = plt.subplots(2, 10, figsize=(20, 5))
fig.suptitle('Sample Images per Class (Train | Test)', fontsize=16, fontweight='bold')

for class_idx in range(10):
    # Find indices
    train_idx = np.where(y_train == class_idx)[0][0]
    test_idx = np.where(y_test == class_idx)[0][0]
    
    # Train sample
    axes[0, class_idx].imshow(X_train[train_idx], cmap='gray')
    axes[0, class_idx].set_title(f'Train: {FMNIST_CLASSES[class_idx]}', fontsize=10)
    axes[0, class_idx].axis('off')
    
    # Test sample
    axes[1, class_idx].imshow(X_test[test_idx], cmap='gray')
    axes[1, class_idx].set_title(f'Test: {FMNIST_CLASSES[class_idx]}', fontsize=10)
    axes[1, class_idx].axis('off')

plt.tight_layout()
plt.show()

### 2.4 Multiple Samples per Class (Variability)

In [ ]:
# Show 5 samples per class to see intra-class variation
n_samples = 5
fig, axes = plt.subplots(10, n_samples, figsize=(15, 25))
fig.suptitle(f'{n_samples} Samples per Class (Training Set)', fontsize=16, fontweight='bold')

for class_idx in range(10):
    indices = np.where(y_train == class_idx)[0][:n_samples]
    for j, idx in enumerate(indices):
        axes[class_idx, j].imshow(X_train[idx], cmap='gray')
        if j == 0:
            axes[class_idx, j].set_ylabel(FMNIST_CLASSES[class_idx], fontsize=12, rotation=0, labelpad=50)
        axes[class_idx, j].axis('off')

plt.tight_layout()
plt.show()

### 2.5 Pixel Intensity Analysis

In [ ]:
# Pixel intensity statistics
print("=== PIXEL INTENSITY STATISTICS ===")
print(f"Train - Mean: {X_train.mean():.2f}, Std: {X_train.std():.2f}")
print(f"Test  - Mean: {X_test.mean():.2f}, Std: {X_test.std():.2f}")
print(f"Train - Min: {X_train.min()}, Max: {X_train.max()}")
print(f"Test  - Min: {X_test.min()}, Max: {X_test.max()}")

# Per-class pixel statistics
class_stats = []
for i in range(10):
    class_imgs = X_train[y_train == i]
    class_stats.append({
        'Class': FMNIST_CLASSES[i],
        'Mean': class_imgs.mean(),
        'Std': class_imgs.std(),
        'Min': class_imgs.min(),
        'Max': class_imgs.max(),
        'Non-zero %': (class_imgs > 0).mean() * 100
    })

stats_df = pd.DataFrame(class_stats)
display(stats_df.round(2))

# Distribution plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(X_train.flatten(), bins=50, color='steelblue', alpha=0.7, edgecolor='black', density=True)
axes[0].set_title('Pixel Intensity Distribution (Train)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Pixel Value (0-255)')
axes[0].set_ylabel('Density')

axes[1].hist(X_test.flatten(), bins=50, color='coral', alpha=0.7, edgecolor='black', density=True)
axes[1].set_title('Pixel Intensity Distribution (Test)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Pixel Value (0-255)')
axes[1].set_ylabel('Density')

plt.tight_layout()
plt.show()

### 2.6 Average Image per Class

In [ ]:
# Compute and visualize average image per class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Average Image per Class (Training Set)', fontsize=16, fontweight='bold')

for class_idx in range(10):
    row = class_idx // 5
    col = class_idx % 5
    class_images = X_train[y_train == class_idx]
    avg_image = np.mean(class_images, axis=0)
    
    im = axes[row, col].imshow(avg_image, cmap='gray')
    axes[row, col].set_title(FMNIST_CLASSES[class_idx], fontsize=12)
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

# Also show standard deviation image (variability)
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Standard Deviation per Class (Variability)', fontsize=16, fontweight='bold')

for class_idx in range(10):
    row = class_idx // 5
    col = class_idx % 5
    class_images = X_train[y_train == class_idx]
    std_image = np.std(class_images, axis=0)
    
    im = axes[row, col].imshow(std_image, cmap='hot')
    axes[row, col].set_title(FMNIST_CLASSES[class_idx], fontsize=12)
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

### 2.7 Pixel Correlation / Covariance

In [ ]:
# Average pixel correlation across all classes
# Flatten images and compute correlation matrix (sample for speed)
sample_size = 5000
indices = np.random.choice(len(X_train), sample_size, replace=False)
X_sample = X_train[indices].reshape(sample_size, -1)

# Compute mean image
mean_img = X_sample.mean(axis=0).reshape(28, 28)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(mean_img, cmap='gray')
plt.title('Mean Image (All Classes)', fontsize=14, fontweight='bold')
plt.axis('off')

# Pixel variance across dataset
var_img = X_sample.var(axis=0).reshape(28, 28)
plt.subplot(1, 2, 2)
plt.imshow(var_img, cmap='hot')
plt.title('Pixel Variance (All Classes)', fontsize=14, fontweight='bold')
plt.axis('off')

plt.tight_layout()
plt.show()

### 2.8 PCA Visualization

In [ ]:
# PCA for dimensionality reduction and visualization
from sklearn.decomposition import PCA

# Sample for speed
sample_size = 5000
indices = np.random.choice(len(X_train), sample_size, replace=False)
X_sample = X_train[indices].reshape(sample_size, -1) / 255.0
y_sample = y_train[indices]

print("Computing PCA...")
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_sample)

print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total explained variance: {pca.explained_variance_ratio_.sum():.2%}")

# Plot
plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_sample, cmap='tab10', alpha=0.6, s=10)
plt.colorbar(scatter, label='Class', ticks=range(10))
plt.title(f'PCA Projection of Fashion MNIST (Explained Variance: {pca.explained_variance_ratio_.sum():.2%})', fontsize=14, fontweight='bold')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})')
plt.tight_layout()
plt.show()

# 3D PCA
pca3 = PCA(n_components=3, random_state=42)
X_pca3 = pca3.fit_transform(X_sample)

fig = px.scatter_3d(
    x=X_pca3[:, 0], y=X_pca3[:, 1], z=X_pca3[:, 2],
    color=y_sample.astype(str),
    labels={'color': 'Class'},
    title=f'3D PCA (Total Variance: {pca3.explained_variance_ratio_.sum():.2%})',
    opacity=0.7
)
fig.show()

### 2.9 t-SNE Visualization (Optional - Slower)

In [ ]:
# t-SNE for better separation visualization
# Uncomment to run (takes ~30-60 seconds)

# from sklearn.manifold import TSNE
# 
# sample_size = 3000
# indices = np.random.choice(len(X_train), sample_size, replace=False)
# X_sample = X_train[indices].reshape(sample_size, -1) / 255.0
# y_sample = y_train[indices]
# 
# print("Computing t-SNE...")
# tsne = TSNE(n_components=2, random_state=42, perplexity=30)
# X_tsne = tsne.fit_transform(X_sample)
# 
# plt.figure(figsize=(10, 8))
# scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_sample, cmap='tab10', alpha=0.6, s=10)
# plt.colorbar(scatter, label='Class', ticks=range(10))
# plt.title('t-SNE Projection of Fashion MNIST', fontsize=14, fontweight='bold')
# plt.xlabel('t-SNE 1')
# plt.ylabel('t-SNE 2')
# plt.tight_layout()
# plt.show()

### 2.10 Class Similarity Analysis

In [ ]:
# Compute average image per class and their pairwise similarities
avg_images = []
for i in range(10):
    class_imgs = X_train[y_train == i]
    avg_images.append(np.mean(class_imgs, axis=0).flatten())

avg_images = np.array(avg_images)  # (10, 784)

# Cosine similarity between class prototypes
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(avg_images)

# Plot similarity matrix
plt.figure(figsize=(10, 8))
sns.heatmap(similarity, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            xticklabels=FMNIST_CLASSES, yticklabels=FMNIST_CLASSES,
            cbar_kws={'label': 'Cosine Similarity'})
plt.title('Class Prototype Similarity (Cosine)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Most similar class pairs
print("Most similar class pairs:")
np.fill_diagonal(similarity, -1)  # Ignore self-similarity
for _ in range(5):
    i, j = np.unravel_index(similarity.argmax(), similarity.shape)
    print(f"  {FMNIST_CLASSES[i]} ↔ {FMNIST_CLASSES[j]}: {similarity[i, j]:.3f}")
    similarity[i, j] = -1

---

## 3️⃣ COMPARISON SUMMARY

In [ ]:
# Summary comparison table
summary = pd.DataFrame({
    'Property': [
        'Dataset Type',
        'Total Samples',
        'Number of Classes',
        'Class Balance',
        'Features',
        'Feature Type',
        'Task Difficulty',
        'Typical Accuracy (CNN)',
        'Typical Accuracy (Tabular)',
        'Has Actual Images'
    ],
    'Clothing Dataset (CSV)': [
        'Tabular/Metadata',
        '5,403',
        '20',
        'Highly Imbalanced (84:1)',
        '4 columns (image_id, sender_id, label, kids)',
        'Categorical + Boolean',
        'Hard (no visual features)',
        'N/A',
        '~15% (with metadata only)',
        '❌ No (only UUID)'
    ],
    'Fashion MNIST': [
        'Images',
        '70,000',
        '10',
        'Perfectly Balanced',
        '784 pixels (28×28)',
        'Grayscale intensity (0-255)',
        'Moderate (visual features)',
        '91-93%',
        '82-86% (with PCA + RF)',
        '✅ Yes (28×28 images)'
    ]
})

display(summary)

---

## 4️⃣ KEY INSIGHTS & RECOMMENDATIONS

### 🔍 Key Findings

#### Clothing Dataset (Metadata-only)
- **Severe class imbalance**: T-Shirt (1,011) vs Skip (12) = 84:1 ratio
- **No visual features**: Only metadata (sender_id, kids flag, image UUID)
- **Sender bias**: Top sender has 150+ images; many senders have 1 image
- **Kids minority**: Only 8.8% kids items
- **Realistic accuracy**: ~15% with metadata only (100% was data leakage)

#### Fashion MNIST (Images)
- **Perfect balance**: 6,000/1,000 per class train/test
- **Rich visual features**: 784 pixels with clear patterns
- **Class similarity**: Pullover/Coat/Shirt similar; Sandal/Sneaker/Ankle boot similar
- **High achievable accuracy**: 91-93% with CNN

### 💡 Recommendations

1. **For Clothing Dataset**: Need actual images! Current metadata insufficient.
2. **For Production**: Combine metadata + image embeddings (ResNet/EfficientNet)
3. **Handle Imbalance**: Use class weights, focal loss, or oversampling
4. **Feature Engineering**: Extract color histograms, texture, shape from images
5. **Data Augmentation**: Critical for minority classes in clothing dataset

---

## 📁 Output Files Generated

Run the Python scripts to generate these files:

`fashion_classification/reports/` - Original pipeline outputs (with leakage)
`fashion_classification/reports_fixed/` - Fixed pipeline outputs (realistic)
`fashion_classification/reports_mnist/` - MNIST pipeline outputs

Each contains:
- PNG visualizations
- Interactive HTML (Plotly)
- CSV reports
- Model comparison tables
- HTML summary report